# 1 — Où se cachent les matrices dans un LLM ?

Objectif : montrer que derrière une phrase de 4 tokens, GPT-2 small (124 M de
paramètres) exécute une cinquantaine de produits matriciels et ~1 GFLOP.

On installe un *hook* PyTorch sur chaque couche linéaire, on lance un forward,
et on lit les formes d'entrée/sortie. Rien de magique : ce sont les mêmes GEMM
que ceux que Strassen et AlphaTensor cherchent à accélérer.

In [ ]:
import sys, torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

PROMPT = "The city of Paris"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
print("paramètres :", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
records = []

def hook(module, inp, out):
    x = inp[0]
    records.append((module.__class__.__name__, tuple(x.shape), tuple(out.shape)))

for _, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Conv1D"):
        mod.register_forward_hook(hook)

inputs = tok(PROMPT, return_tensors="pt")
with torch.no_grad():
    model(**inputs)

print("tokens :", inputs["input_ids"].shape[1], "| couches linéaires :", len(records))

In [ ]:
total = 0
for i, (cls, ins, outs) in enumerate(records[:8], 1):
    flops = 2 * ins[0] * ins[1] * ins[-1] * outs[-1]
    total += flops
    print(f"{i:>2} {cls:<8} {str(ins):<18} -> {str(outs):<18} {flops/1e6:8.2f} MFLOPs")
print("...")
print("total affiché pour 8 couches :", f"{total/1e9:.3f} GFLOPs")

## Ce qu'il faut retenir

- GPT-2 small a **12 blocs**, chacun avec 4 GEMM (attention QKV, projection de sortie,
  FFN up, FFN down) → 48 GEMM par token, plus la tête de sortie.
- Un LLM de 124 M de paramètres fait ~1 GFLOP pour 4 tokens. Un modèle de 400 G
  de paramètres (GPT-4 classe) en fait des milliers.
- **Toute** l'industrie de l'accélération IA est une industrie de multiplication
  matricielle. D'où l'intérêt d'un exposant plus petit que 3.